In [1]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Load unseen-species model data
path = '../3-unseen_species_index/unseen_species_model/unseen_species_model_bayesian.csv'
df_cultura_original = pd.read_csv(path, index_col = [0])

df_cultura_original['N_est'] = df_cultura_original['f0'] + df_cultura_original['f1'] + df_cultura_original['f2']
df_cultura_original['N_est_min']  =  df_cultura_original['min_f0'] + df_cultura_original['f1'] + df_cultura_original['f2']
df_cultura_original['N_est_max']  =  df_cultura_original['max_f0'] + df_cultura_original['f1'] + df_cultura_original['f2']

df_cultura = df_cultura_original.groupby(['region_name', 'decade']).agg({'N_est': 'sum', 'N_est_min':'sum', 'N_est_max':'sum', 'individual_wikidata_id':'count'}).reset_index()
df_cultura = df_cultura.rename(columns={"individual_wikidata_id": "n_cultural_producers"})

df_save = df_cultura.copy()
df_save = df_save.rename(columns={
    'N_est': 'cultural_production',
    'N_est_min': 'cultural_production_min',
    'N_est_max': 'cultural_production_max'
})

# Filter Greek World to stop at year 500
df_save = df_save[~((df_save['region_name'] == 'Greek World') & (df_save['decade'] > 500))]

# Add macro_region_name mapping
macro_region_mapping = {
    'Greek World': 'Ancient Mediterranean',
    'Latin World': 'Ancient Mediterranean',
    'Chinese world': 'Asia',
    'Indian world': 'Asia',
    'Japan': 'Asia',
    'Korea': 'Asia',
    'Northern China': 'Asia',
    'Northern Japan': 'Asia',
    'Southern China': 'Asia',
    'Southern Japan': 'Asia',
    'Central Europe': 'Eastern Europe',
    'East Slavic': 'Eastern Europe',
    'Arabic world': 'Middle-East and Africa (MENA)',
    'Persian world': 'Middle-East and Africa (MENA)',
    'France': 'Western Europe',
    'German world': 'Western Europe',
    'Italy': 'Western Europe',
    'Low countries': 'Western Europe',
    'Nordic countries': 'Western Europe',
    'Northwestern Europe': 'Western Europe',
    'Portugal': 'Western Europe',
    'Southwestern Europe': 'Western Europe',
    'Spain': 'Western Europe',
    'United Kingdom': 'Western Europe',
}

df_save['macro_region_name'] = df_save['region_name'].map(macro_region_mapping)

# Reorder columns
df_save = df_save[['macro_region_name', 'region_name', 'decade', 'n_cultural_producers', 
                   'cultural_production', 'cultural_production_min', 'cultural_production_max']]

df_save

,macro_region_name,region_name,decade,n_cultural_producers,cultural_production,cultural_production_min,cultural_production_max
0,Middle-East and Africa (MENA),Arabic world,-610,1,5.529170,3.469453,12.010166
1,Middle-East and Africa (MENA),Arabic world,-400,1,2.611448,2.070892,3.533070
2,Middle-East and Africa (MENA),Arabic world,-360,3,14.825990,11.441811,20.637703
3,Middle-East and Africa (MENA),Arabic world,-330,1,6.061846,4.709635,8.383012
4,Middle-East and Africa (MENA),Arabic world,-310,1,6.022250,4.717134,8.197015
...,...,...,...,...,...,...,...
2393,Western Europe,United Kingdom,1840,936,12298.471136,11394.352286,13351.587402
2394,Western Europe,United Kingdom,1850,909,12344.101231,11411.735253,13418.204576
2395,Western Europe,United Kingdom,1860,1144,15480.038693,14291.622840,16848.799422
2396,Western Europe,United Kingdom,1870,912,11764.644742,10868.216912,12784.160433


In [2]:
# Create a research-oriented Excel file with metadata sheet (Maddison-style)
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill

# Create workbook
wb = Workbook()

# ============================================
# SHEET 1: INFO / README
# ============================================
ws_info = wb.active
ws_info.title = "Info"

# Styling
title_font = Font(bold=True, size=14)
header_font = Font(bold=True, size=11)
normal_font = Font(size=10)

# Dataset information
info_content = [
    ["CULTURAL PRODUCTION INDEX DATABASE"],
    [""],
    ["AUTHORS"],
    ["Charles de Dampierre", "ENS Paris"],
    ["", "charlesdedampierre@gmail.com"],
    [""],
    ["CITATION"],
    ["Please cite as:"],
    ["de Dampierre, C., Kestemont, M., Karsdorp, F., Chevalier, C., Harper, K., Huillery, E.,"],
    ["Koyama, M., Voth, J., Bolt, J., Luiten van Zanden, J., Bassino, J.-P., Chen, T., Kase, V.,"],
    ["Kumon, Y., Morris, I., Nakabayashi, M., Shatzmiller, M., Xu, T., Yazdani, K., Thouzeau, V.,"],
    ["& Baumard, N. (2026). Cultural Production Reveals Transitions to Sustained Human"],
    ["Development in both European and Non-European Societies. Working paper."],
    ["https://hal.science/hal-05452762"],
    [""],
    ["DESCRIPTION"],
    ["This database provides estimates of cultural production across world regions from antiquity"],
    ["to the modern era. The estimates are derived from biographical data in Wikidata, using an"],
    ["unseen species model to correct for historical selection bias."],
    [""],
    ["VARIABLE DEFINITIONS"],
    [""],
    ["macro_region_name", "Broad geographic grouping (e.g., Western Europe, Asia)"],
    ["region_name", "Geographic region (modern country or historical region)"],
    ["decade", "Amount of cultural production for one decade"],
    ["n_cultural_producers", "Number of observed cultural producers in the data"],
    ["cultural_production", "Unseen species model estimate of cultural producers (point estimate)"],
    ["cultural_production_min", "Lower bound of the 94% credible interval"],
    ["cultural_production_max", "Upper bound of the 94% credible interval"],
    [""],
    ["GEOGRAPHIC COVERAGE"],
    ["The database covers the following macro-regions:"],
]

# Add macro-regions list
macro_regions = sorted(df_save['macro_region_name'].unique())
for macro_region in macro_regions:
    info_content.append(["- " + macro_region])

info_content.extend([
    [""],
    ["TEMPORAL COVERAGE"],
    [f"From: {df_save['decade'].min()} to {df_save['decade'].max()}"],
    ["Unit: Decades"],
])

# Write info content
for row_idx, row in enumerate(info_content, 1):
    for col_idx, value in enumerate(row, 1):
        cell = ws_info.cell(row=row_idx, column=col_idx, value=value)
        if row_idx == 1:
            cell.font = title_font
        elif value in ["AUTHORS", "CITATION", "DESCRIPTION", 
                       "VARIABLE DEFINITIONS", "GEOGRAPHIC COVERAGE", "TEMPORAL COVERAGE"]:
            cell.font = header_font
        else:
            cell.font = normal_font

# Adjust column widths for info sheet
ws_info.column_dimensions['A'].width = 25
ws_info.column_dimensions['B'].width = 70

# ============================================
# SHEET 2: REGIONS
# ============================================
ws_regions = wb.create_sheet("Regions")

# Full regions data with modern_countries and spans
regions_data = [
    {"macro_region_name": "Ancient Mediterranean", "region_name": "Greek World", "modern_countries": "Ukraine, Albania, Montenegro, Kosovo, Turkey, Greece, Bulgaria, Romania, France (until 300BC), Italy (until 300BC), Spain (until 300BC), Libya, Egypt, Israel, Palestine, Lebanon, Syrian Arab Republic, Jordan, Cyprus, Iraq", "spans": "-820 to 1880"},
    {"macro_region_name": "Ancient Mediterranean", "region_name": "Latin World", "modern_countries": "Tunisia, Algeria, Morocco, Romania, Croatia, Serbia, Bosnia and Herzegovina, Slovenia, France, United Kingdom, Germany, Switzerland, Austria, Spain, Portugal, Italy", "spans": "-320 to 510"},
    {"macro_region_name": "Asia", "region_name": "Chinese world", "modern_countries": "China, Mongolia, Taiwan", "spans": "-680 to 1880"},
    {"macro_region_name": "Asia", "region_name": "Northern China", "modern_countries": "China (north of latitude 33°)", "spans": "-470 to 1880"},
    {"macro_region_name": "Asia", "region_name": "Southern China", "modern_countries": "China (south of latitude 33°)", "spans": "-680 to 1880"},
    {"macro_region_name": "Asia", "region_name": "Indian world", "modern_countries": "India, Pakistan, Bangladesh, Sri Lanka, Nepal", "spans": "-530 to 1880"},
    {"macro_region_name": "Asia", "region_name": "Japan", "modern_countries": "Japan", "spans": "10 to 1880"},
    {"macro_region_name": "Asia", "region_name": "Northern Japan", "modern_countries": "Japan (east of longitude 138°)", "spans": "1270 to 1880"},
    {"macro_region_name": "Asia", "region_name": "Southern Japan", "modern_countries": "Japan (west of longitude 138°)", "spans": "690 to 1880"},
    {"macro_region_name": "Asia", "region_name": "Korea", "modern_countries": "Korea", "spans": "450 to 1880"},
    {"macro_region_name": "Eastern Europe", "region_name": "Central Europe", "modern_countries": "Latvia, Estonia, Slovakia, Lithuania, Czechia, Poland, Hungary", "spans": "840 to 1880"},
    {"macro_region_name": "Eastern Europe", "region_name": "East Slavic", "modern_countries": "Belarus, Russian Federation, Ukraine", "spans": "860 to 1880"},
    {"macro_region_name": "Middle-East and Africa (MENA)", "region_name": "Arabic world", "modern_countries": "Tunisia, Algeria, Morocco, Libya, Egypt, Palestine, Israel, Lebanon, Syrian Arab Republic, Jordan, Iraq, Kuwait, Oman, United Arab Emirates, Saudi Arabia, Bahrain, Yemen", "spans": "-610 to 1880"},
    {"macro_region_name": "Middle-East and Africa (MENA)", "region_name": "Persian world", "modern_countries": "Iran, Afghanistan, Kyrgyzstan, Uzbekistan, Turkmenistan, Azerbaijan", "spans": "-360 to 1880"},
    {"macro_region_name": "Western Europe", "region_name": "France", "modern_countries": "France", "spans": "550 to 1880"},
    {"macro_region_name": "Western Europe", "region_name": "German world", "modern_countries": "Germany, Switzerland, Austria", "spans": "570 to 1880"},
    {"macro_region_name": "Western Europe", "region_name": "Italy", "modern_countries": "Italy", "spans": "500 to 1880"},
    {"macro_region_name": "Western Europe", "region_name": "Low countries", "modern_countries": "Netherlands, Belgium", "spans": "640 to 1880"},
    {"macro_region_name": "Western Europe", "region_name": "Nordic countries", "modern_countries": "Denmark, Norway, Sweden, Finland, Iceland", "spans": "940 to 1880"},
    {"macro_region_name": "Western Europe", "region_name": "Northwestern Europe", "modern_countries": "Sweden, Norway, Denmark, France, Ireland, United Kingdom, Iceland, Finland, Belgium, Netherlands, Germany, Switzerland, Austria", "spans": "-100 to 1880"},
    {"macro_region_name": "Western Europe", "region_name": "Portugal", "modern_countries": "Portugal", "spans": "1020 to 1880"},
    {"macro_region_name": "Western Europe", "region_name": "Spain", "modern_countries": "Spain", "spans": "580 to 1880"},
    {"macro_region_name": "Western Europe", "region_name": "Southwestern Europe", "modern_countries": "Portugal, Italy, Spain", "spans": "-620 to 1880"},
    {"macro_region_name": "Western Europe", "region_name": "United Kingdom", "modern_countries": "Ireland, United Kingdom", "spans": "540 to 1880"},
]

regions_df = pd.DataFrame(regions_data)

# Write headers
region_headers = ['macro_region_name', 'region_name', 'modern_countries', 'spans']
for col_idx, header in enumerate(region_headers, 1):
    cell = ws_regions.cell(row=1, column=col_idx, value=header)
    cell.font = header_font
    cell.fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")

# Write data
for row_idx, row in enumerate(regions_df.values, 2):
    for col_idx, value in enumerate(row, 1):
        ws_regions.cell(row=row_idx, column=col_idx, value=value)

# Adjust column widths
ws_regions.column_dimensions['A'].width = 35
ws_regions.column_dimensions['B'].width = 20
ws_regions.column_dimensions['C'].width = 80
ws_regions.column_dimensions['D'].width = 15

# ============================================
# SHEET 3: DATA
# ============================================
ws_data = wb.create_sheet("Data")

# Round numerical columns to 2 decimal places
df_save_rounded = df_save.copy()
numeric_cols = ['cultural_production', 'cultural_production_min', 'cultural_production_max']
df_save_rounded[numeric_cols] = df_save_rounded[numeric_cols].round(2)

# Write headers
headers = list(df_save_rounded.columns)
for col_idx, header in enumerate(headers, 1):
    cell = ws_data.cell(row=1, column=col_idx, value=header)
    cell.font = header_font
    cell.fill = PatternFill(start_color="D9E1F2", end_color="D9E1F2", fill_type="solid")

# Write data
for row_idx, row in enumerate(df_save_rounded.values, 2):
    for col_idx, value in enumerate(row, 1):
        ws_data.cell(row=row_idx, column=col_idx, value=value)

# Adjust column widths for data sheet
ws_data.column_dimensions['A'].width = 30
ws_data.column_dimensions['B'].width = 20
ws_data.column_dimensions['C'].width = 10
ws_data.column_dimensions['D'].width = 20
ws_data.column_dimensions['E'].width = 20
ws_data.column_dimensions['F'].width = 25
ws_data.column_dimensions['G'].width = 25

# Save
output_path = "cultural_production_index_database.xlsx"
wb.save(output_path)
print(f"Excel file saved: {output_path}")

Excel file saved: cultural_production_index_database.xlsx
